# Hướng dẫn chạy dự án Dubbing trên Google Colab

Notebook này giúp bạn tự động clone repository `https://github.com/kinyias/dubbing.git`, thiết lập môi trường, cài đặt phụ thuộc, tự động đăng ký thiết bị lấy `DUANJU_DEVICE_ID` và `DUANJU_INSTALL_ID` động, sau đó khởi chạy Backend FastAPI với **Cloudflare Tunnel (trycloudflare.com)**.

## 1. Clone Repository từ GitHub

In [ ]:
# Clone dự án từ GitHub về môi trường Colab
%cd /content
!rm -rf /content/dubbing
!git clone https://github.com/kinyias/dubbing.git
%cd /content/dubbing

## 2. Kiểm tra GPU và Cài đặt System Dependencies (FFmpeg, Node.js & Cloudflared)

In [ ]:
# Kiểm tra GPU được cấp phát
!nvidia-smi

# Cài đặt FFmpeg, Node.js và Cloudflared CLI
!apt-get update -qq
!apt-get install -y ffmpeg nodejs npm wget -qq

# Tải và cài đặt cloudflared
!wget -q https://github.com/cloudflare/cloudflare-tunnel-downloads/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## 3. Cài đặt Node.js & Python Dependencies

In [ ]:
%cd /content/dubbing

# Cài đặt Node.js dependencies cho node_helper.js trong backend
!cd backend && npm install

# Cài đặt Python dependencies
!pip install -r requirements.txt

## 4. Tự động Đăng ký Thiết bị lấy Device ID & Install ID động và Khởi tạo `.env` & `settings.json`

In [ ]:
import json
import os
import sys

# 1. Chạy module device_register để lấy DUANJU_DEVICE_ID và DUANJU_INSTALL_ID động
backend_path = os.path.abspath("backend")
sys.path.insert(0, backend_path)
sys.path.insert(0, os.path.join(backend_path, "service", "liushen"))

device_id = ""
install_id = ""
try:
    from service.liushen.device_register import device_register
    print("📲 Đang gửi request đăng ký thiết bị tự động...")
    reg_res = device_register()
    device_id = reg_res.get("device_id", "")
    install_id = reg_res.get("install_id", "")
    print(f"✅ Đăng ký thiết bị thành công! Device ID: {device_id} | Install ID: {install_id}")
except Exception as e:
    print(f"⚠️ Không thể tự động đăng ký thiết bị: {e}")

# 2. Tạo file .env với ID vừa lấy được
env_content = f"""DUANJU_DEVICE_ID={device_id}
DUANJU_INSTALL_ID={install_id}
DUANJU_PLATFORM=android
APP_PORT=8000
OPEN_BROWSER=0
FLASK_DEBUG=0
FFMPEG_BIN=ffmpeg
"""
with open(".env", "w", encoding="utf-8") as f:
    f.write(env_content)
print("✅ Đã khởi tạo file .env thành công!")

# 3. Tạo file backend/settings.json từ settings.example.json
settings_example_path = "backend/settings.example.json"
settings_path = "backend/settings.json"
if os.path.exists(settings_example_path):
    with open(settings_example_path, "r", encoding="utf-8") as f:
        settings = json.load(f)
    settings["ffmpegPath"] = "ffmpeg"
    with open(settings_path, "w", encoding="utf-8") as f:
        json.dump(settings, f, indent=4, ensure_ascii=False)
    print("✅ Đã khởi tạo backend/settings.json thành công!")

## 5. Kiểm tra thử nghiệm Pipeline (CLI Test)

In [ ]:
%cd /content/dubbing
!python run_pipeline.py --help

## 6. Khởi chạy Backend Server (FastAPI) & Mở Public URL qua Cloudflare Tunnel (trycloudflare.com)

In [ ]:
import subprocess
import time
import re

# 1. Khởi chạy FastAPI Backend ở nền trên port 8000
backend_proc = subprocess.Popen(["python", "backend/main.py"])
time.sleep(3)

# 2. Mở Public Tunnel với trycloudflare
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("🚀 Đang khởi tạo Cloudflare Tunnel...")
for line in iter(tunnel_proc.stdout.readline, ""):
    print(line, end="")
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            print("\n==================================================")
            print(f"🎉 PUBLIC API URL: {match.group(0)}")
            print("==================================================\n")